In [1]:
!pip -q install google-genai pypdf sentence-transformers chromadb gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [3]:
!pip -q install pypdf sentence-transformers chromadb gradio google-genai --no-deps

In [4]:
!pip -q install google-auth requests websockets


In [5]:
import os
from getpass import getpass
from google import genai

API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("✅ Gemini connected successfully!")

Enter your Gemini API key: ··········
✅ Gemini connected successfully!


In [6]:
hospital_text = """
CITY CARE HOSPITAL

GENERAL INFORMATION

City Care Hospital is a multi-specialty hospital providing general medical
services and emergency care.

DEPARTMENTS

1. Cardiology
Provides consultation and treatment related to heart and cardiovascular conditions.

2. Neurology
Provides consultation for disorders related to the brain, nerves, and nervous system.

3. Orthopedics
Provides treatment and consultation for bones, joints, muscles, and sports injuries.

4. Pediatrics
Provides medical care for infants, children, and teenagers.

5. General Medicine
Provides consultation and treatment for common illnesses and general health problems.

OPD TIMINGS

Monday to Saturday: 9:00 AM to 5:00 PM.
Sunday: 9:00 AM to 1:00 PM.

EMERGENCY SERVICES

The emergency department is available 24 hours a day, 7 days a week.

APPOINTMENTS

Patients can book an appointment through the hospital reception.
Patients should carry a valid identification document and previous medical records
when visiting for a consultation.

VISITING HOURS

General visiting hours are from 4:00 PM to 7:00 PM.
Visitors should follow hospital safety and visitor guidelines.

FACILITIES

The hospital provides:
- Pharmacy
- Laboratory services
- Radiology services
- Ambulance services
- Emergency care
- Inpatient rooms
- Outpatient consultation

CONTACT INFORMATION

Hospital Reception: 044-1234-5678
Emergency: 108
Email: info@citycarehospital.example

IMPORTANT NOTICE

This information is for demonstration of a RAG system.
For actual medical advice, patients should consult qualified healthcare professionals.
"""

print("✅ Hospital information created!")
print(hospital_text[:500])


✅ Hospital information created!

CITY CARE HOSPITAL

GENERAL INFORMATION

City Care Hospital is a multi-specialty hospital providing general medical
services and emergency care.

DEPARTMENTS

1. Cardiology
Provides consultation and treatment related to heart and cardiovascular conditions.

2. Neurology
Provides consultation for disorders related to the brain, nerves, and nervous system.

3. Orthopedics
Provides treatment and consultation for bones, joints, muscles, and sports injuries.

4. Pediatrics
Provides medical care for 


In [7]:
def create_chunks(text, chunk_size=500, overlap=50):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = create_chunks(hospital_text)

print(f"✅ Created {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i + 1} ---")
    print(chunk)
    print()


✅ Created 4 chunks

--- Chunk 1 ---
CITY CARE HOSPITAL

GENERAL INFORMATION

City Care Hospital is a multi-specialty hospital providing general medical
services and emergency care.

DEPARTMENTS

1. Cardiology
Provides consultation and treatment related to heart and cardiovascular conditions.

2. Neurology
Provides consultation for disorders related to the brain, nerves, and nervous system.

3. Orthopedics
Provides treatment and consultation for bones, joints, muscles, and sports injuries.

4. Pediatrics
Provides medical care for

--- Chunk 2 ---
njuries.

4. Pediatrics
Provides medical care for infants, children, and teenagers.

5. General Medicine
Provides consultation and treatment for common illnesses and general health problems.

OPD TIMINGS

Monday to Saturday: 9:00 AM to 5:00 PM.
Sunday: 9:00 AM to 1:00 PM.

EMERGENCY SERVICES

The emergency department is available 24 hours a day, 7 days a week.

APPOINTMENTS

Patients can book an appointment through the hospital reception.
Patie

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks).tolist()

print("✅ Embeddings created successfully!")
print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

In [9]:
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="hospital_information"
)

ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.upsert(
    ids=ids,
    documents=chunks,
    embeddings=embeddings
)

print("✅ Hospital information stored in ChromaDB!")
print("Total chunks:", collection.count())

✅ Hospital information stored in ChromaDB!
Total chunks: 4


In [10]:
def hospital_rag(question):

    # Convert question into an embedding
    question_embedding = embedding_model.encode(
        [question]
    ).tolist()

    # Search ChromaDB for relevant information
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )

    retrieved_chunks = results["documents"][0]

    # Combine retrieved information
    context = "\n\n".join(retrieved_chunks)

    # Create prompt for Gemini
    prompt = f"""
You are a Hospital Information Assistant.

Answer the user's question using ONLY the information
provided in the context below.

Do not invent information.

If the answer is not available in the context, say:
"The information is not available in the hospital information."

Keep the answer clear and concise.

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text


print("✅ Hospital RAG function created successfully!")

✅ Hospital RAG function created successfully!


In [12]:
question = "What are the OPD timings?"

answer = hospital_rag(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: What are the OPD timings?

Answer:
The OPD timings are:
- Monday to Saturday: 9:00 AM to 5:00 PM
- Sunday: 9:00 AM to 1:00 PM


In [13]:
import gradio as gr

def run_hospital_rag(question):
    if not question.strip():
        return "Please enter a question."

    return hospital_rag(question)


demo = gr.Interface(
    fn=run_hospital_rag,
    inputs=gr.Textbox(
        label="🏥 Ask a Hospital Question",
        placeholder="Example: What are the visiting hours?"
    ),
    outputs=gr.Markdown(
        label="💡 Answer"
    ),
    title="🏥 Hospital Information RAG Assistant",
    description="Ask questions about hospital departments, timings, services, facilities, appointments, and other information."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://966426a913153e220a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
